# **SQLD 학습 도우미: LangChain + Gemini + Gradio 기반 RAG 에이전트**
수업 노트북 `Langchain_rag_geminiAPI.ipynb`를 바탕으로, 주제를 **SQLD 시험**으로 바꾼 버전입니다.


# 0. 환경 설정

In [1]:
!pip install langchain langchain-text-splitters langchain-core -q
!pip install -U "langchain[google-genai]" -q
!pip install -U langchain-google-genai -q
!pip install -U gradio -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.3/573.3 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.9/81.9 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.9/267.9 kB 21.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 4.1 MB/s eta 0:00:00


In [2]:
import os
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('SQLDPracticeGradio')
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

In [4]:
from langchain.chat_models import init_chat_model
MODEL_NAME = "google_genai:gemini-3.5-flash-lite"

model = init_chat_model(MODEL_NAME)
model

ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.6.10', 'langchain': '1.4.5', 'langchain-google-genai': '4.4.2'}}, profile={'name': 'Gemini 3.5 Flash Lite', 'release_date': '2026-07-21', 'last_updated': '2026-07-21', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True, 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high'], 'reasoning_effort_default': 'minimal'}, google_api_key=SecretStr('**********'), model='gemini-3.5-flash-lite', temperature=None, client=<google.genai.client.Client object at 0x7a845fd8ea50>, default_metadata=(), model_kwargs=

In [5]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
embeddings

GoogleGenerativeAIEmbeddings(client=<google.genai.client.Client object at 0x7a845fa4fed0>, model='models/gemini-embedding-001', task_type=None, google_api_key=SecretStr('**********'), credentials=None, vertexai=None, project=None, location=None, base_url=None, additional_headers=None, client_args=None, api_version=None, request_options=None, output_dimensionality=None)

In [6]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)
vector_store

# 1. 인덱싱
## 문서 불러오기: SQLD 학습 노트
웹페이지를 가져오는 대신, 주제별 SQLD 학습 노트를 노트북 안에 직접 작성해 지식 베이스로 사용합니다.

In [7]:
from langchain_core.documents import Document

sqld_notes = {

"시험 개요": """
SQLD(SQL 개발자)는 한국데이터산업진흥원(K-DATA)에서 시행하는 국가공인 자격시험입니다. 공식 사이트는 dataq.or.kr 입니다.
시험은 두 과목으로 구성됩니다.
- 1과목 데이터 모델링의 이해: 10문항, 20점
- 2과목 SQL 기본 및 활용: 40문항, 80점
총 50문항, 100점 만점, 시험 시간은 90분입니다. 객관식이 대부분이고 단답형 문제가 일부 포함됩니다.
합격 기준은 총점 60점 이상이면서 각 과목의 점수가 40% 이상이어야 합니다. 한 과목이라도 40% 미만이면 과락으로 불합격입니다.
시험 규정은 바뀔 수 있으므로 최신 내용은 반드시 공식 사이트에서 확인해야 합니다.
""",

"데이터 모델링 기초": """
데이터 모델링은 개념적 모델링, 논리적 모델링, 물리적 모델링의 3단계로 진행됩니다.
개념적 모델링은 업무 관점에서 핵심 엔터티와 관계를 도출하며 특정 DBMS에 종속되지 않습니다.
논리적 모델링은 키, 속성, 관계를 상세히 정의하고 정규화를 수행하며 역시 DBMS에 독립적입니다.
물리적 모델링은 실제 DBMS에 맞춰 테이블, 컬럼, 인덱스를 설계하며 성능과 저장 효율을 고려합니다.
데이터 모델은 사물(엔터티), 속성, 관계의 세 가지 요소로 표현됩니다.
데이터 모델링 시 유의점은 중복 배제, 비유연성 방지(유연성 확보), 일관성 유지입니다.
3단계 스키마 구조는 외부 스키마, 개념 스키마, 내부 스키마로 이루어지며 데이터 독립성을 제공합니다.
ERD(개체 관계 다이어그램)는 데이터 모델을 그림으로 표현하는 도구입니다.
""",

"엔터티, 속성, 관계": """
엔터티는 업무에서 관리해야 하는 대상입니다. 엔터티가 되려면 인스턴스가 2개 이상이어야 하고, 식별자로 구분할 수 있어야 하며, 속성을 가지고 다른 엔터티와 관계를 맺어야 합니다.
유무형에 따른 분류는 유형 엔터티, 개념 엔터티, 사건 엔터티입니다.
발생 시점에 따른 분류는 기본 엔터티, 중심 엔터티, 행위 엔터티입니다.
속성은 더 이상 쪼갤 수 없는 최소의 데이터 단위입니다. 특성에 따라 기본 속성, 설계 속성, 파생 속성으로 나누고, 구성 방식에 따라 단일값 속성, 복합 속성, 다중값 속성으로 나눕니다.
관계의 차수(Cardinality)는 1:1, 1:N, M:N이 있으며, 선택성에 따라 필수 관계와 선택 관계로 나뉩니다. M:N 관계는 교차(연관) 엔터티를 만들어 해소합니다.
식별 관계는 부모의 기본키가 자식의 기본키의 일부가 되는 관계이며 ERD에서 실선으로 표시합니다.
비식별 관계는 부모의 기본키가 자식의 일반 속성(외래키)이 되는 관계이며 점선으로 표시합니다.
""",

"식별자": """
주식별자가 갖춰야 할 특징은 유일성, 최소성, 불변성, 존재성(NULL이 될 수 없음)입니다.
식별자의 분류는 다음과 같습니다. 대표성 여부에 따라 주식별자와 보조식별자, 스스로 생성 여부에 따라 내부식별자와 외부식별자, 속성의 수에 따라 단일식별자와 복합식별자, 대체 여부에 따라 본질식별자와 인조식별자입니다.
인조식별자는 본질식별자가 너무 복잡할 때 일련번호처럼 인위적으로 만든 식별자입니다.
""",

"정규화": """
정규화는 데이터 중복을 제거하고 이상 현상(삽입 이상, 갱신 이상, 삭제 이상)을 막기 위해 테이블을 분리하는 과정입니다.
제1정규형은 모든 속성이 원자값을 가져야 하며 반복 그룹이 없어야 합니다.
제2정규형은 제1정규형을 만족하면서 복합 기본키의 일부에만 종속되는 부분 함수 종속을 제거한 상태입니다.
제3정규형은 제2정규형을 만족하면서 기본키가 아닌 속성이 다른 일반 속성에 종속되는 이행 함수 종속을 제거한 상태입니다.
BCNF는 모든 결정자가 후보키인 상태입니다.
제4정규형은 다치 종속을 제거하고, 제5정규형은 조인 종속을 제거합니다.
반정규화는 조회 성능을 높이기 위해 의도적으로 중복을 허용하는 것입니다. 종류는 테이블 반정규화(병합, 분할, 추가), 컬럼 반정규화(중복 컬럼, 파생 컬럼 추가), 관계 반정규화(관계 추가)가 있습니다. 조회는 빨라지지만 데이터 일관성과 입력, 수정, 삭제 성능은 나빠질 수 있습니다.
""",

"NULL 처리": """
NULL은 값을 알 수 없거나 해당되지 않는 상태를 뜻합니다. 0이나 공백 문자열과는 다릅니다.
NULL이 포함된 산술 연산 결과는 항상 NULL입니다. 예를 들어 100 + NULL은 NULL입니다.
NULL은 =, <> 같은 비교 연산으로 비교할 수 없으며 IS NULL, IS NOT NULL을 사용해야 합니다.
집계 함수는 NULL 값을 무시합니다. 다만 COUNT(*)는 NULL을 포함한 전체 행 수를 셉니다. COUNT(컬럼)은 NULL을 세지 않습니다.
NULL 처리 함수: NVL(값, 대체값)은 Oracle, ISNULL(값, 대체값)은 SQL Server에서 사용합니다. COALESCE(a, b, c, ...)는 표준 함수로 NULL이 아닌 첫 번째 값을 반환합니다. NULLIF(a, b)는 a와 b가 같으면 NULL, 다르면 a를 반환합니다.
ORDER BY에서 Oracle은 NULL을 가장 큰 값으로 취급해 오름차순에서 맨 뒤에 놓고, SQL Server는 NULL을 가장 작은 값으로 취급해 오름차순에서 맨 앞에 놓습니다.
""",

"SELECT와 WHERE": """
SQL의 논리적 실행 순서는 FROM, WHERE, GROUP BY, HAVING, SELECT, ORDER BY 입니다.
SELECT가 WHERE보다 나중에 실행되므로 컬럼 별칭은 WHERE에서 쓸 수 없고 ORDER BY에서는 쓸 수 있습니다.
연산자 종류: 비교 연산자(=, <>, !=, <, >), SQL 연산자(BETWEEN a AND b는 양 끝값 포함, IN, LIKE의 % 와 _, IS NULL), 논리 연산자(AND, OR, NOT).
연산 우선순위는 괄호, NOT, AND, OR 순서입니다.
ORDER BY는 기본이 오름차순(ASC)이며 내림차순은 DESC를 씁니다. 가장 마지막에 실행됩니다.
DISTINCT는 중복된 행을 제거합니다. 문자열 연결은 Oracle에서 ||, SQL Server에서 +를 쓰며 CONCAT() 함수는 둘 다 사용할 수 있습니다.
""",

"단일행 함수": """
문자 함수: UPPER, LOWER, LENGTH(SQL Server는 LEN), SUBSTR(SQL Server는 SUBSTRING), INSTR, REPLACE, TRIM, LTRIM, RTRIM, LPAD, RPAD, CONCAT.
숫자 함수: ABS, SIGN, MOD(a, b)는 나머지, CEIL은 올림, FLOOR는 내림, ROUND(값, 자릿수)는 반올림, TRUNC(값, 자릿수)는 버림, POWER, SQRT.
날짜 함수: SYSDATE(Oracle), GETDATE()(SQL Server), ADD_MONTHS, EXTRACT, TO_CHAR(날짜, 형식), TO_DATE(문자, 형식).
형변환은 명시적 형변환(TO_CHAR, TO_NUMBER, TO_DATE, CAST, CONVERT)과 DBMS가 자동으로 처리하는 암시적 형변환으로 나뉩니다.
CASE 표현식: CASE WHEN 조건 THEN 결과 ... ELSE 결과 END 형태이며 표준 문법으로 모든 DBMS에서 사용할 수 있습니다. DECODE(식, 비교값, 결과, ..., 기본값)는 Oracle 전용입니다.
""",

"GROUP BY와 집계 함수": """
집계 함수에는 COUNT, SUM, AVG, MIN, MAX가 있고 STDDEV, VARIANCE도 있습니다. 모두 NULL을 무시하며 COUNT(*)만 예외입니다.
GROUP BY는 행을 그룹으로 묶으며, SELECT 절의 집계 함수가 아닌 컬럼은 반드시 GROUP BY에 있어야 합니다.
HAVING은 그룹화한 결과를 걸러내고, WHERE는 그룹화 이전의 행을 걸러냅니다. 집계 함수는 HAVING에서는 쓸 수 있지만 WHERE에서는 쓸 수 없습니다.
ROLLUP(a, b)은 오른쪽부터 하나씩 줄여 가며 소계를 만들고 마지막에 총계를 만듭니다. 컬럼 순서가 결과에 영향을 줍니다.
CUBE(a, b)는 가능한 모든 조합의 소계와 총계를 만듭니다. 컬럼 순서는 결과에 영향을 주지 않습니다.
GROUPING SETS((a), (b))는 지정한 그룹별 집계만 만듭니다.
GROUPING(컬럼) 함수는 해당 컬럼의 소계 행이면 1, 아니면 0을 반환합니다.
""",

"조인": """
INNER JOIN(등가 조인)은 두 테이블에서 값이 일치하는 행만 반환합니다.
NATURAL JOIN은 이름이 같은 모든 컬럼을 자동으로 조인 조건으로 사용하며, 조인 컬럼에는 테이블 별칭을 붙일 수 없습니다.
JOIN ... USING (컬럼)은 지정한 같은 이름의 컬럼으로 조인합니다.
CROSS JOIN은 카티션 곱으로 결과 행 수가 두 테이블 행 수의 곱(m x n)이 됩니다. 조인 조건이 빠졌을 때도 이렇게 됩니다.
OUTER JOIN은 LEFT, RIGHT, FULL이 있으며 일치하지 않는 행도 한쪽 또는 양쪽에서 유지합니다. Oracle 전통 문법에서는 데이터가 없을 수 있는 쪽에 (+)를 붙입니다.
SELF JOIN은 같은 테이블을 서로 다른 별칭으로 자기 자신과 조인하는 것입니다.
비등가 조인은 = 이외의 연산자(BETWEEN, <, > 등)로 조인하는 방식입니다.
N개의 테이블을 조인하려면 최소 N-1개의 조인 조건이 필요합니다.
""",

"서브쿼리": """
서브쿼리는 다른 SQL 문 안에 포함된 SELECT 문이며 반드시 괄호로 묶습니다.
위치에 따른 분류: 스칼라 서브쿼리(SELECT 절, 값 하나 반환), 인라인 뷰(FROM 절), 중첩 서브쿼리(WHERE, HAVING 절).
반환 행 수에 따른 분류: 단일행 서브쿼리(=, <, >, <> 사용), 다중행 서브쿼리(IN, ANY, ALL, EXISTS 사용), 다중 컬럼 서브쿼리.
ANY는 값 중 하나라도 조건을 만족하면 참이고, ALL은 모든 값이 조건을 만족해야 참입니다.
연관 서브쿼리는 메인 쿼리의 컬럼을 참조하며 메인 쿼리의 행마다 실행됩니다. EXISTS와 함께 자주 쓰이며 서브쿼리 결과가 한 행이라도 있으면 참입니다.
WHERE 절의 서브쿼리에서는 ORDER BY를 사용할 수 없고, 인라인 뷰에서는 사용할 수 있습니다.
""",

"집합 연산자": """
UNION은 결과를 합치고 중복을 제거하며 정렬도 수행합니다. UNION ALL은 중복을 제거하지 않고 정렬도 하지 않아 더 빠릅니다.
INTERSECT는 두 쿼리 결과에 모두 있는 행만 반환합니다.
MINUS(Oracle) 또는 EXCEPT(SQL Server, 표준)는 첫 번째 쿼리에는 있고 두 번째 쿼리에는 없는 행을 반환합니다.
규칙: 두 쿼리의 컬럼 수가 같고 대응하는 컬럼의 데이터 타입이 호환되어야 합니다. 결과의 컬럼 이름은 첫 번째 쿼리를 따릅니다. ORDER BY는 맨 마지막에 한 번만 쓸 수 있습니다.
""",

"윈도우 함수": """
문법: 함수() OVER (PARTITION BY 컬럼 ORDER BY 컬럼 [ROWS 또는 RANGE 윈도우 절]).
순위 함수: RANK는 동점이면 같은 순위를 주고 다음 순위를 건너뜁니다(예: 1, 1, 3). DENSE_RANK는 순위를 건너뛰지 않습니다(1, 1, 2). ROW_NUMBER는 동점이어도 고유한 일련번호를 줍니다(1, 2, 3).
집계 윈도우 함수: SUM, AVG, COUNT, MAX, MIN에 OVER 절을 붙여 사용하며, ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW로 누적 합계를 구할 수 있습니다.
행 순서 함수: FIRST_VALUE, LAST_VALUE, LAG(컬럼, n)은 n번째 이전 행의 값, LEAD(컬럼, n)은 n번째 이후 행의 값을 가져옵니다.
비율 함수: RATIO_TO_REPORT, PERCENT_RANK, CUME_DIST, NTILE(n).
윈도우 함수는 SELECT 절과 ORDER BY 절에서만 사용할 수 있고 WHERE나 GROUP BY에서는 사용할 수 없습니다.
""",

"Top N 쿼리와 계층형 질의": """
Top N 쿼리: Oracle의 ROWNUM은 ORDER BY보다 먼저 부여되므로, 정렬된 상위 N개를 구하려면 인라인 뷰에서 먼저 정렬한 뒤 바깥에서 ROWNUM으로 거릅니다. Oracle 12c 이상에서는 FETCH FIRST n ROWS ONLY도 사용할 수 있습니다. SQL Server는 SELECT TOP n을 쓰며 WITH TIES를 붙이면 동점자도 포함합니다.
계층형 질의(Oracle): START WITH는 시작(루트) 조건, CONNECT BY PRIOR 부모키 = 자식키는 순방향 전개, LEVEL은 깊이, ORDER SIBLINGS BY는 형제 노드 정렬, CONNECT_BY_ISLEAF는 말단 여부, SYS_CONNECT_BY_PATH는 경로 표시, NOCYCLE은 순환 방지입니다.
CONNECT BY PRIOR 사원번호 = 관리자번호는 부모에서 자식으로 내려가는 순방향 전개입니다. CONNECT BY 사원번호 = PRIOR 관리자번호는 자식에서 부모로 올라가는 역방향 전개입니다.
PIVOT은 행의 값을 컬럼으로 바꾸고, UNPIVOT은 컬럼을 행으로 바꿉니다.
정규 표현식(Oracle): REGEXP_LIKE, REGEXP_SUBSTR, REGEXP_REPLACE, REGEXP_INSTR, REGEXP_COUNT.
""",

"DML과 트랜잭션": """
DML에는 INSERT, UPDATE, DELETE, MERGE가 있습니다. MERGE는 조건에 따라 입력 또는 수정을 한 번에 처리합니다.
TCL: COMMIT은 변경 내용을 확정하고, ROLLBACK은 취소하며, SAVEPOINT는 되돌아갈 지점을 지정합니다(ROLLBACK TO 저장점이름).
트랜잭션의 ACID 특성은 원자성, 일관성, 격리성, 지속성입니다.
DELETE는 행을 삭제하며 DML이라 ROLLBACK이 가능합니다. TRUNCATE는 모든 행을 빠르게 삭제하고 저장 공간을 반환하며 DDL이라 Oracle에서는 ROLLBACK이 불가능합니다. DROP은 테이블 구조와 데이터를 모두 삭제합니다.
Oracle에서는 DDL을 실행하면 자동으로 COMMIT이 됩니다. SQL Server는 기본이 자동 커밋이며 명시적 트랜잭션은 BEGIN TRAN으로 시작해 COMMIT 또는 ROLLBACK으로 끝냅니다.
""",

"DDL, 제약조건, 뷰, DCL": """
DDL에는 CREATE, ALTER(ADD, MODIFY, DROP COLUMN), DROP, RENAME, TRUNCATE가 있습니다.
데이터 타입: CHAR는 고정 길이로 남는 부분을 공백으로 채우고, VARCHAR 또는 VARCHAR2는 가변 길이이며, NUMBER는 숫자, DATE는 날짜입니다.
제약조건: PRIMARY KEY는 유일하고 NOT NULL이며 테이블당 하나만 만들 수 있습니다. UNIQUE는 중복을 허용하지 않지만 NULL은 허용합니다. NOT NULL은 NULL을 금지하고, CHECK는 값의 조건을 지정하며, FOREIGN KEY는 참조 무결성을 보장합니다.
외래키의 삭제 옵션: CASCADE는 자식 행도 함께 삭제, SET NULL은 자식의 외래키를 NULL로 변경, RESTRICT 또는 NO ACTION은 삭제를 막습니다.
DROP TABLE ... CASCADE CONSTRAINTS는 해당 테이블을 참조하는 외래키 제약조건도 함께 삭제합니다.
뷰는 쿼리로 정의한 가상 테이블이며 장점은 독립성, 편리성, 보안성입니다.
DCL: GRANT는 권한을 부여하고 REVOKE는 권한을 회수합니다. ROLE은 여러 권한을 묶어 관리합니다.
""",
}

docs = [
    Document(page_content=text.strip(), metadata={"source": "SQLD 학습 노트", "topic": topic})
    for topic, text in sqld_notes.items()
]
print(f"학습 노트 {len(docs)}개를 불러왔습니다.")
print(docs[0].page_content[:300])

학습 노트 16개를 불러왔습니다.
SQLD(SQL 개발자)는 한국데이터산업진흥원(K-DATA)에서 시행하는 국가공인 자격시험입니다. 공식 사이트는 dataq.or.kr 입니다.
시험은 두 과목으로 구성됩니다.
- 1과목 데이터 모델링의 이해: 10문항, 20점
- 2과목 SQL 기본 및 활용: 40문항, 80점
총 50문항, 100점 만점, 시험 시간은 90분입니다. 객관식이 대부분이고 단답형 문제가 일부 포함됩니다.
합격 기준은 총점 60점 이상이면서 각 과목의 점수가 40% 이상이어야 합니다. 한 과목이라도 40% 미만이면 과락으로 불합격입니다.
시험 규정은 바


## 문서 분할

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)
all_splits = text_splitter.split_documents(docs)
print(f"Split study notes into {len(all_splits)} sub-documents.")

Split study notes into 16 sub-documents.


## 문서 저장 (벡터 스토어)

In [9]:
document_ids = vector_store.add_documents(documents=all_splits)
print(document_ids[:3])

['baed2e91-7046-4103-9d36-bb4b44fbe2fd', '80c51492-5822-47d1-ac19-276613e3630b', '7fbf7963-88d9-4d8f-9486-57573fda3ced']


# 2. 검색과 답변 생성
## RAG 에이전트

In [10]:
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """질문에 답하는 데 도움이 되는 SQLD 학습 노트를 검색합니다."""
    retrieved_docs = vector_store.similarity_search(query, k=3)
    serialized = "\n\n".join(
        f"주제: {doc.metadata.get('topic')}\n내용: {doc.page_content}"
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

In [11]:
from langchain.agents import create_agent

tools = [retrieve_context]
prompt = (
    "당신은 SQLD(SQL 개발자) 시험을 준비하는 학생을 돕는 친절한 튜터입니다. "
    "답변하기 전에 반드시 검색 도구로 학습 노트를 먼저 찾아보세요. "
    "항상 한국어로 답하고, 필요하면 짧은 예시나 SQL 문장을 곁들여 쉽고 간결하게 설명하세요. "
    "학습 노트에 없는 내용이면 노트에 없다고 먼저 밝히고, 일반 지식으로 신중하게 답하세요. "
    "시험 규정과 세부 사항은 공식 사이트(dataq.or.kr)에서 확인하라고 안내하세요."
)
agent = create_agent(model, tools, system_prompt=prompt)

## 간단 테스트 (Gradio 전에)

In [12]:
query = "RANK, DENSE_RANK, ROW_NUMBER의 차이는 무엇인가요?"

for event in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

================================ Human Message =================================

RANK, DENSE_RANK, ROW_NUMBER의 차이는 무엇인가요?
================================== Ai Message ==================================

[]
Tool Calls:
  retrieve_context (call_472135)
 Call ID: call_472135
  Args:
    query: RANK DENSE_RANK ROW_NUMBER 차이
================================= Tool Message =================================
Name: retrieve_context

주제: 윈도우 함수
내용: 문법: 함수() OVER (PARTITION BY 컬럼 ORDER BY 컬럼 [ROWS 또는 RANGE 윈도우 절]).
순위 함수: RANK는 동점이면 같은 순위를 주고 다음 순위를 건너뜁니다(예: 1, 1, 3). DENSE_RANK는 순위를 건너뛰지 않습니다(1, 1, 2). ROW_NUMBER는 동점이어도 고유한 일련번호를 줍니다(1, 2, 3).
집계 윈도우 함수: SUM, AVG, COUNT, MAX, MIN에 OVER 절을 붙여 사용하며, ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW로 누적 합계를 구할 수 있습니다.
행 순서 함수: FIRST_VALUE, LAST_VALUE, LAG(컬럼, n)은 n번째 이전 행의 값, LEAD(컬럼, n)은 n번째 이후 행의 값을 가져옵니다.
비율 함수: RATIO_TO_REPORT, PERCENT_RANK, CUME_DIST, NTILE(n).
윈도우 함수는 SELECT 절과 ORDER BY 절에서만 사용할 수 있고 WHERE나 GROUP BY에서는 사용할 수 없습니다.

주제: Top N 

# 3. Gradio 채팅 화면 (외부 공개 링크)


In [ ]:
import gradio as gr

STUDENT = "2555041_(ESCOBEDO JIMENEZ ANGIE ALEJANDRA)"
def extract_text(content):
    if isinstance(content, str):
        return content
    return "".join(
        block.get("text", "") if isinstance(block, dict) else str(block)
        for block in content
    )

def chat(message, history):
    result = agent.invoke({"messages": [{"role": "user", "content": message}]})
    return extract_text(result["messages"][-1].content)

demo = gr.ChatInterface(
    fn=chat,
    title=f"SQLD 학습 도우미 ({STUDENT})",
    description="SQLD 시험에 대해 질문해 보세요! (데이터 모델링의 이해 / SQL 기본 및 활용)",
    examples=[
        "RANK와 DENSE_RANK의 차이는?",
        "WHERE와 HAVING의 차이는?",
        "제1정규형, 제2정규형, 제3정규형을 설명해 주세요.",
        "SQLD 시험은 몇 문항이고 합격 기준은 어떻게 되나요?",
    ],
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4c6fa93b5bcd48e826.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
